# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
import pandas as pd
import numpy as np

# Load dataset
dataset = pd.read_csv('work/outputs/dataset.csv')

# Feature columns
feature_cols = [
    'impressions_90d', 'clicks_90d', 'ctr_90d', 'avg_position_90d',
    'sessions_90d', 'pageviews_90d', 'engaged_sessions_90d',
    'organic_sessions_90d', 'impressions_last30', 'impressions_first60',
    'momentum_pct', 'active_days_90d', 'has_ga4_data', 'has_momentum'
]

# Columns that must never appear as features
leak_cols = [
    'impressions_mar', 'impressions_apr', 'pct_change',
    'trend_direction', 'trend_pct'
]

print(f"Dataset loaded: {dataset.shape}")
print(f"Feature columns: {len(feature_cols)}")
print("=== FEATURE VECTOR ===")
print(f"Shape: {dataset[feature_cols].shape}")
print(f"\nFirst 5 rows:")
print(dataset[feature_cols].head().to_string())
print("=== FEATURE TYPES ===")
for col in feature_cols:
    dtype = dataset[col].dtype
    n_unique = dataset[col].nunique()
    print(f"  {col:30s}  dtype={str(dtype):10s}  unique={n_unique:,}")

Dataset loaded: (116114, 17)
Feature columns: 14
=== FEATURE VECTOR ===
Shape: (116114, 14)

First 5 rows:
   impressions_90d  clicks_90d  ctr_90d  avg_position_90d  sessions_90d  pageviews_90d  engaged_sessions_90d  organic_sessions_90d  impressions_last30  impressions_first60  momentum_pct  active_days_90d  has_ga4_data  has_momentum
0           7574.0         5.0   0.0660             12.09           0.0            0.0                   0.0                   0.0               822.0               6752.0        -75.65               90             0             1
1            528.0         0.0   0.0000             52.75           0.0            0.0                   0.0                   0.0               200.0                328.0         21.95               89             0             1
2          46802.0        32.0   0.0684              5.55           0.0            0.0                   0.0                   0.0             13919.0              32883.0        -15.34               

## 1. Build the feature vector

The feature vector is built from the 90-day feature window (Jan–Mar 2026).
All features are aggregated at the `client_hash_id` + `content_hash_id` level.
No feature uses any data from April 2026 or later.

Two types of engineered features are included:

- **Structural flags:** `has_ga4_data` and `has_momentum` — identify rows where
  GA4 data is unavailable or where the first 60 days had zero impressions.
  These prevent the model from confusing "no data" with "zero activity."

- **Momentum signal:** `momentum_pct` compares the last 30 days of the feature
  window against the average of the first 60 days. A positive value means
  traffic is accelerating; negative means it is decelerating.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Every feature below existed before April 2026 — no feature requires
knowledge of the label window to compute.

| Feature | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| `impressions_90d` | Total search impressions Jan–Mar | No nulls | Yes |
| `clicks_90d` | Total search clicks Jan–Mar | No nulls | Yes |
| `ctr_90d` | Click-through rate Jan–Mar | No nulls | Yes |
| `avg_position_90d` | Mean search rank Jan–Mar | No nulls, floored at 1.0 | Yes |
| `sessions_90d` | Total GA4 sessions Jan–Mar | Zero-filled, flagged by `has_ga4_data` | Yes |
| `pageviews_90d` | Total GA4 pageviews Jan–Mar | Zero-filled, flagged by `has_ga4_data` | Yes |
| `engaged_sessions_90d` | Total GA4 engaged sessions Jan–Mar | Zero-filled, flagged by `has_ga4_data` | Yes |
| `organic_sessions_90d` | Total organic sessions Jan–Mar | Zero-filled, flagged by `has_ga4_data` | Yes |
| `impressions_last30` | Impressions in March only | No nulls | Yes |
| `impressions_first60` | Impressions in Jan–Feb only | No nulls | Yes |
| `momentum_pct` | Traffic trend direction Jan–Mar | Zero-filled when `has_momentum=0`, capped at 99th pct | Yes |
| `active_days_90d` | Days with GSC data in window | No nulls | Yes |
| `has_ga4_data` | 1 if client has GA4 | No nulls | Yes |
| `has_momentum` | 1 if first 60 days had impressions | No nulls | Yes |

In [3]:
print("=== MISSING VALUE AUDIT ===")
nulls = dataset[feature_cols].isnull().sum()
if nulls.sum() == 0:
    print("No null values in any feature column.")
else:
    print("Columns with nulls:")
    print(nulls[nulls > 0])

print(f"\nGA4 structural zero-fill verification:")
ga4_cols = ['sessions_90d', 'pageviews_90d', 'engaged_sessions_90d', 'organic_sessions_90d']
no_ga4 = dataset['has_ga4_data'] == 0
for col in ga4_cols:
    all_zero = (dataset.loc[no_ga4, col] == 0).all()
    print(f"  {col:30s}  all zero when has_ga4_data=0: {all_zero}")

print(f"\nMomentum structural zero-fill verification:")
no_momentum = dataset['has_momentum'] == 0
all_zero = (dataset.loc[no_momentum, 'momentum_pct'] == 0).all()
print(f"  momentum_pct zero when has_momentum=0: {all_zero}")

=== MISSING VALUE AUDIT ===
No null values in any feature column.

GA4 structural zero-fill verification:
  sessions_90d                    all zero when has_ga4_data=0: True
  pageviews_90d                   all zero when has_ga4_data=0: True
  engaged_sessions_90d            all zero when has_ga4_data=0: True
  organic_sessions_90d            all zero when has_ga4_data=0: True

Momentum structural zero-fill verification:
  momentum_pct zero when has_momentum=0: True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

A feature leaks if it encodes information about the label that would not be
available at prediction time.

Three categories of leakage risk for this project:

1. **Label-derived columns:** columns computed from the same March/April
   impression comparison used to build `is_declining_label`.

2. **Future window columns:** any column that aggregates data from April 2026
   or later — the period we are trying to predict.

3. **High correlation with label:** features that are suspiciously predictive
   may indicate indirect leakage. We check correlation as a secondary signal.

In [4]:
print("=== DIRECT LEAKAGE CHECK ===")
print("Columns that must NOT appear in the dataset:")
print()
for col in leak_cols:
    present = col in dataset.columns
    status = "PRESENT — CRITICAL RISK" if present else "absent — OK"
    print(f"  {col:30s}  {status}")

print(f"\nResult: {'LEAKAGE DETECTED — FIX BEFORE MODELING' if any(c in dataset.columns for c in leak_cols) else 'No direct leakage detected.'}")

print("=== CORRELATION WITH LABEL ===")
print("Checking for suspiciously high feature-label correlations.")
print("Values above 0.8 warrant investigation.\n")

correlations = dataset[feature_cols].corrwith(
    dataset['is_declining_label']
).abs().sort_values(ascending=False)

for feat, corr in correlations.items():
    flag = " <-- INVESTIGATE" if corr > 0.8 else ""
    print(f"  {feat:30s}  {corr:.4f}{flag}")
print("=== TEMPORAL BOUNDARY CHECK ===")
print("Verifying no feature uses data from April 2026 or later.\n")

# impressions_last30 must be March data only — check it is <= impressions_90d
boundary_ok = (dataset['impressions_last30'] <= dataset['impressions_90d']).all()
print(f"impressions_last30 <= impressions_90d: {boundary_ok}")

# impressions_first60 + impressions_last30 must equal impressions_90d
total_check = (
    (dataset['impressions_first60'] + dataset['impressions_last30'])
    == dataset['impressions_90d']
).all()
print(f"first60 + last30 == impressions_90d:   {total_check}")

print(f"\nTemporal boundary is clean: {boundary_ok and total_check}")

=== DIRECT LEAKAGE CHECK ===
Columns that must NOT appear in the dataset:

  impressions_mar                 absent — OK
  impressions_apr                 absent — OK
  pct_change                      absent — OK
  trend_direction                 absent — OK
  trend_pct                       absent — OK

Result: No direct leakage detected.
=== CORRELATION WITH LABEL ===
Checking for suspiciously high feature-label correlations.
Values above 0.8 warrant investigation.

  has_momentum                    0.1394
  ctr_90d                         0.1216
  has_ga4_data                    0.1065
  active_days_90d                 0.0917
  clicks_90d                      0.0758
  impressions_last30              0.0586
  impressions_90d                 0.0563
  organic_sessions_90d            0.0485
  impressions_first60             0.0483
  sessions_90d                    0.0336
  pageviews_90d                   0.0319
  engaged_sessions_90d            0.0295
  momentum_pct                    0

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


| Column | Reason |
|---|---|
| `impressions_mar` | Numerator of the label formula — knowing March impressions at prediction time is fine, but this column was constructed alongside the label and carries its exact values. Excluded to avoid any accidental leakage path. |
| `impressions_apr` | Future data — April impressions are what we are predicting. Using them as a feature would give the model perfect information. |
| `pct_change` | Computed directly from `impressions_mar` and `impressions_apr` — encodes the label by definition. |
| `trend_direction` | Categorical encoding of `pct_change` — same leakage as above. |
| `trend_pct` | Percentage form of `pct_change` — same leakage as above. |
| `content_hash_id` | Pseudonymized identifier — not a predictive signal. Used only for row identification. |
| `client_hash_id` | Pseudonymized identifier — not a predictive signal. Used only for grouped train/test split. |

In [5]:
print("=== EXCLUSION VERIFICATION ===")

excluded_all = leak_cols + ['content_hash_id', 'client_hash_id']

print("Excluded columns and their status in dataset:\n")
for col in excluded_all:
    present = col in dataset.columns
    if col in ['content_hash_id', 'client_hash_id']:
        status = "present as context — OK (not in feature_cols)" if present else "absent"
    else:
        status = "PRESENT — REMOVE" if present else "absent — OK"
    print(f"  {col:30s}  {status}")

print(f"\nAll leak columns excluded from features: "
      f"{not any(c in feature_cols for c in leak_cols)}")

print(f"\n=== FEATURE VECTOR FINAL STATUS ===")
print(f"Features:          {len(feature_cols)}")
print(f"Null values:       {dataset[feature_cols].isnull().sum().sum()}")
print(f"Direct leakage:    {any(c in dataset.columns for c in leak_cols)}")
high_corr = (dataset[feature_cols].corrwith(dataset['is_declining_label']).abs() > 0.8).any()
print(f"High correlation:  {high_corr}")
print(f"\nFeature vector status: CLEAN — READY FOR MODELING")

=== EXCLUSION VERIFICATION ===
Excluded columns and their status in dataset:

  impressions_mar                 absent — OK
  impressions_apr                 absent — OK
  pct_change                      absent — OK
  trend_direction                 absent — OK
  trend_pct                       absent — OK
  content_hash_id                 present as context — OK (not in feature_cols)
  client_hash_id                  present as context — OK (not in feature_cols)

All leak columns excluded from features: True

=== FEATURE VECTOR FINAL STATUS ===
Features:          14
Null values:       0
Direct leakage:    False
High correlation:  False

Feature vector status: CLEAN — READY FOR MODELING


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.